# Here we try to implement a conditional Variational AutoEncoder for solving inverse problem of quantum circuit generation

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np

from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam

import qiskit
from qiskit.quantum_info import Operator

In [2]:
#!pip install torch

In [45]:
# Conditional Variational Autoencoder (cVAE) for Multi-Solution Inverse Problem
# Recovering 18 angles (6x3) from a 4x4 complex matrix

# ============================
# Helper to Flatten Complex Matrix
# ============================
def complex_matrix_to_tensor(matrix):
    """
    Converts 4x4 complex matrix to 32D real vector (real and imaginary parts).
    Input: torch.complex64 or np.array of shape (4, 4)
    Output: torch.tensor of shape (32,)
    """
    return torch.cat([matrix.real.flatten(), matrix.imag.flatten()], dim=-1)

# ============================
# Conditional VAE Components
# ============================
class Encoder(nn.Module):
    def __init__(self, input_dim=32, output_dim=18, latent_dim=64):
        super().__init__()
        self.fc1 = nn.Linear(input_dim + output_dim, 128)
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

    def forward(self, x_cond, y):
        x = torch.cat([x_cond, y], dim=1)
        h = F.relu(self.fc1(x))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

class Decoder(nn.Module):
    def __init__(self, input_dim=32, latent_dim=32, output_dim=18):
        super().__init__()
        self.fc1 = nn.Linear(input_dim + latent_dim, 128)
        self.fc2 = nn.Linear(128, output_dim)

    def forward(self, x_cond, z):
        x = torch.cat([x_cond, z], dim=1)
        h = F.relu(self.fc1(x))
        return self.fc2(h)

class CVAE(nn.Module):
    def __init__(self, cond_dim=32, output_dim=18, latent_dim=64):
        super().__init__()
        self.encoder = Encoder(cond_dim, output_dim, latent_dim)
        self.decoder = Decoder(cond_dim, latent_dim, output_dim)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.1 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x_cond, y=None):
        if y is not None:
            mu, logvar = self.encoder(x_cond, y)
            z = self.reparameterize(mu, logvar)
            y_pred = self.decoder(x_cond, z)
            return y_pred, mu, logvar
        else:
            # Inference mode (sampling)
            z = torch.randn(x_cond.size(0), self.encoder.fc_mu.out_features).to(x_cond.device)
            y_pred = self.decoder(x_cond, z)
            return y_pred

# ============================
# Loss Function
# ============================
def cvae_loss(recon_y, y, mu, logvar):
    recon_loss = F.mse_loss(recon_y, y, reduction='mean')
    kl_div = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / y.size(0)
    return recon_loss + kl_div, recon_loss, kl_div


In [10]:
def get_matrix_for_double_cz_block(angles):
    qc = qiskit.QuantumCircuit(2)
    qc.u(angles[0], angles[1], angles[2], 0)
    qc.u(angles[3], angles[4], angles[5], 1)
    qc.cz(0, 1)
    qc.u(angles[6], angles[7], angles[8], 0)
    qc.u(angles[9], angles[10], angles[11], 1)
    qc.cz(0, 1)
    qc.u(angles[12], angles[13], angles[14], 0)
    qc.u(angles[15], angles[16], angles[17], 1)
    op = Operator(qc)
    matrix = op.data
    return matrix

In [11]:
def generate_dataset(N=100000):
    X = []  # main_operator (flattened)
    Y = []  # angles
    for _ in range(N):
        angles = np.random.uniform(0, 2*np.pi, size=18)
        main_operator = get_matrix_for_double_cz_block(angles)  
        X.append(np.concatenate([main_operator.real.flatten(), main_operator.imag.flatten()]))
        Y.append(angles)
    
    return torch.tensor(X, dtype=torch.float32), torch.tensor(Y, dtype=torch.float32)


In [6]:
# Generate data
X, Y = generate_dataset()
dataset = TensorDataset(X, Y)

#print(loader)


C:\Users\Oleg\AppData\Local\Temp\ipykernel_30612\3807250040.py:10: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:257.)
  return torch.tensor(X, dtype=torch.float32), torch.tensor(Y, dtype=torch.float32)


In [46]:
loader = DataLoader(dataset, batch_size=100000, shuffle=True)
# Initialize model
model = CVAE()
optimizer = Adam(model.parameters(), lr=1e-3)

# Training loop
for epoch in range(150):
    #print(epoch)
    total_loss = 0
    for x_batch, y_batch in loader:
        #print(len(y_batch))
        y_pred, mu, logvar = model(x_batch, y_batch)
        loss, recon, kl = cvae_loss(y_pred, y_batch, mu, logvar)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: Loss = {total_loss:.3f}")

Epoch 1: Loss = 29.739
Epoch 2: Loss = 25.517
Epoch 3: Loss = 22.347
Epoch 4: Loss = 19.989
Epoch 5: Loss = 18.251
Epoch 6: Loss = 16.978
Epoch 7: Loss = 16.043
Epoch 8: Loss = 15.350
Epoch 9: Loss = 14.825
Epoch 10: Loss = 14.410
Epoch 11: Loss = 14.062
Epoch 12: Loss = 13.747
Epoch 13: Loss = 13.451
Epoch 14: Loss = 13.159
Epoch 15: Loss = 12.875
Epoch 16: Loss = 12.587
Epoch 17: Loss = 12.303
Epoch 18: Loss = 12.023
Epoch 19: Loss = 11.742
Epoch 20: Loss = 11.472
Epoch 21: Loss = 11.205
Epoch 22: Loss = 10.942
Epoch 23: Loss = 10.684
Epoch 24: Loss = 10.428
Epoch 25: Loss = 10.180
Epoch 26: Loss = 9.931
Epoch 27: Loss = 9.691
Epoch 28: Loss = 9.454
Epoch 29: Loss = 9.214
Epoch 30: Loss = 8.975
Epoch 31: Loss = 8.738
Epoch 32: Loss = 8.500
Epoch 33: Loss = 8.279
Epoch 34: Loss = 8.042
Epoch 35: Loss = 7.820
Epoch 36: Loss = 7.605
Epoch 37: Loss = 7.385
Epoch 38: Loss = 7.173
Epoch 39: Loss = 6.962
Epoch 40: Loss = 6.772
Epoch 41: Loss = 6.572
Epoch 42: Loss = 6.397
Epoch 43: Loss = 6

In [35]:
def complex_matrix_to_tensor(matrix):
    """
    Converts a 4x4 complex matrix (NumPy or PyTorch) to a 32D real PyTorch tensor.
    Real and imaginary parts are concatenated.
    
    Args:
        matrix: shape (4, 4), dtype=complex64 or complex128 (NumPy array or torch.Tensor)
    
    Returns:
        torch.FloatTensor of shape (32,)
    """
    if isinstance(matrix, np.ndarray):
        # Convert real and imaginary parts to PyTorch tensors
        real = torch.from_numpy(matrix.real).float()
        imag = torch.from_numpy(matrix.imag).float()
    elif isinstance(matrix, torch.Tensor):
        # Use .real and .imag directly
        real = matrix.real.float()
        imag = matrix.imag.float()
    else:
        raise TypeError("Input must be a NumPy ndarray or a PyTorch tensor with complex dtype.")

    return torch.cat([real.flatten(), imag.flatten()], dim=0)  # shape: (32,)

In [42]:
model.eval()

angles = np.random.uniform(0, 2*np.pi, size=18)
main_op_matr = get_matrix_for_double_cz_block(angles)
main_op = np.concatenate([main_op_matr.real.flatten(), main_op_matr.imag.flatten()])

x_input = complex_matrix_to_tensor(main_op_matr).unsqueeze(0)  # shape: (1, 32)

samples = []
for _ in range(10):  # generate 10 samples
    with torch.no_grad():
        angles_pred = model(x_input)
        angles_array = angles_pred.detach().cpu().numpy()
        pred_matr  = get_matrix_for_double_cz_block(angles_array[0])
        # shape: (1, 24)
        diff = np.linalg.norm(main_op_matr - pred_matr)
        print(diff)
    samples.append(angles_pred.squeeze().numpy())


2.8087397992749548
2.7300847255992045
2.960033717328173
2.810984682557345
2.844879611723457
2.8483998102132904
2.801797820268071
2.743754976144375
2.8276235885436796
2.9071433094208556


In [50]:
angles_pred[0]

tensor(2.6829)

In [51]:
numpy_array = angles_pred.detach().cpu().numpy()

In [53]:
numpy_array[0]

array([2.6829188, 2.938578 , 3.3724716, 2.8640685, 3.1306643, 2.3605816,
       2.912266 , 2.9452512, 3.1377833, 3.006136 , 3.0880265, 3.18501  ,
       2.9221108, 2.9624712, 3.0012758, 3.0906436, 2.7876878, 3.3056362],
      dtype=float32)